> **Note:** This notebook reuses patterns inspired by the Azure AI Gateway Labs (https://github.com/Azure-Samples/AI-Gateway).

# 🛡️ Citadel Hosted Agent with AGT Governance

## Overview

This notebook builds and deploys a **governed Foundry Hosted Agent** that integrates the
[Agent Governance Toolkit (AGT)](https://github.com/microsoft/agent-governance-toolkit)
to enforce policy-based tool gating **server-side** inside the container.

| Component | Detail |
|-----------|--------|
| **Agent Framework** | Microsoft Agent Framework (`agent-framework`) |
| **Governance** | AGT Middleware: `CapabilityGuardMiddleware` + `GovernancePolicyMiddleware` + `AuditTrailMiddleware` |
| **Protocol** | Responses API (hosted on port 8088) |
| **Model** | `gpt-4.1` via APIM BYO Gateway connection (`Hub-HR-ChatAgent-DEV-LLM/gpt-4.1`) |
| **Tools** | `get_weather` (allowed), `get_current_time` (allowed), `send_email` (blocked) |
| **Telemetry** | OpenTelemetry → Application Insights |
| **Container Build** | `az acr build` (cloud build — no local Docker required) |

## How AGT Governance Works

```
User Request → Foundry → Agent Container
                              |
                    [LLM picks tool]
                              |
                    [AGT Middleware checks policy]
                         /           \\\\
                      ALLOW            DENY
                        |                 |
              Execute tool     Block + return error
                        |                 |
                   Response to user
```

## Prerequisites

- Notebook 7 completed (basic hosted agent deployed successfully)
- Spoke Foundry resources deployed (`workshop/scripts/deploy-spoke-foundry.ps1`)
- BYO Gateway connection `Hub-HR-ChatAgent-DEV-LLM` created (notebook 3)
- Azure CLI logged in (`az login`)

<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

Read environment values from the `azd` environment and configure variables for this notebook.

In [ ]:
# Read environment values from azd
import subprocess, json

def azd_get_value(key):
    """Read a value from the current azd environment."""
    result = subprocess.run(
        ["azd", "env", "get-value", key],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"Failed to read {key}: {result.stderr.strip()}")
    return result.stdout.strip()

env_governance_hub_resource_group = azd_get_value("AZURE_RESOURCE_GROUP")
env_location = azd_get_value("AZURE_LOCATION")

env_foundry_subscription_id = azd_get_value("AZURE_SUBSCRIPTION_ID")
env_foundry_resource_group  = azd_get_value("SPOKE_RESOURCE_GROUP")
env_foundry_account_name    = azd_get_value("SPOKE_AI_FOUNDRY_ACCOUNT_NAME")
env_foundry_project_name    = azd_get_value("SPOKE_AI_FOUNDRY_PROJECT_NAME")
env_acr_name                = azd_get_value("SPOKE_ACR_NAME")
env_acr_login_server        = azd_get_value("SPOKE_ACR_LOGIN_SERVER")

print(f"Citadel Hub resource group: {env_governance_hub_resource_group}")
print(f"Location: {env_location}")
print(f"Foundry: {env_foundry_account_name} (Project: {env_foundry_project_name})")
print(f"Spoke RG: {env_foundry_resource_group}")
print(f"ACR: {env_acr_name} ({env_acr_login_server})")

In [ ]:
import os
import sys, json, requests, time
sys.path.insert(1, '../shared')  # add the shared directory to the Python path
import utils

# ============================================================================
# 🔧 GOVERNED AGENT CONFIGURATION
# ============================================================================
governed_agent_name       = "citadel-governed-agent"
governed_agent_image_tag  = "v1"
governed_agent_model      = "Hub-HR-ChatAgent-DEV-LLM/gpt-4.1"  # BYO Gateway connection/model

# Derived values
governed_agent_image      = f"{env_acr_login_server}/{governed_agent_name}:{governed_agent_image_tag}"
foundry_project_endpoint  = f"https://{env_foundry_account_name}.services.ai.azure.com/api/projects/{env_foundry_project_name}"

utils.print_info(f"Agent name: {governed_agent_name}")
utils.print_info(f"Image: {governed_agent_image}")
utils.print_info(f"Model: {governed_agent_model}")
utils.print_info(f"Foundry endpoint: {foundry_project_endpoint}")

<a id='1'></a>
### 1️⃣ Verify Azure CLI Login

Ensure you are logged in to Azure CLI and the correct subscription is active.

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Generate Governed Agent Source Files

This cell writes the governed agent application code into `workshop/governed-agent/`.

The agent has three tools:
- **`get_current_time(timezone)`** — Returns current time (ALLOWED by CapabilityGuardMiddleware)
- **`get_weather(location)`** — Returns simulated weather (ALLOWED by CapabilityGuardMiddleware)
- **`send_email(to, body)`** — Sends an email (BLOCKED — not in allowed_tools list)

**AGT Integration (Tutorial 34 pattern):**
- `CapabilityGuardMiddleware` — allow-list of permitted tool names (function middleware)
- `GovernancePolicyMiddleware` — YAML-based rules for PII/input blocking (agent middleware)
- `AuditTrailMiddleware` — logs every interaction for compliance (agent middleware)
- Middleware registered on `AgentKernel` and passed to the `Agent` constructor

In [ ]:
import pathlib

agent_dir = pathlib.Path("governed-agent")
agent_dir.mkdir(exist_ok=True)

# ---- governance.py: Self-contained governance module (AGT-compatible pattern) ----
# Implements the same semantics as AGT CapabilityGuardMiddleware + GovernancePolicyMiddleware
# without depending on external packages that may not yet be published to PyPI.
governance_py = r'''"""
Minimal AGT-compatible governance module for Foundry Hosted Agents.

Implements:
- CapabilityGuard: tool allow-list (blocks unauthorized tool calls)
- PolicyEngine: regex-based input blocking (PII, sensitive data)
- AuditLog: structured logging of all governance decisions

Pattern based on: https://microsoft.github.io/agent-governance-toolkit/tutorials/34-maf-integration/
"""
import re
import logging
from dataclasses import dataclass, field
from datetime import datetime, timezone

logger = logging.getLogger("agt.governance")


@dataclass
class GovernanceDecision:
    """Result of a governance check."""
    allowed: bool
    tool_name: str
    reason: str
    timestamp: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())


class CapabilityGuard:
    """Blocks tool calls not in the allowed_tools list."""

    def __init__(self, allowed_tools: list[str]):
        self.allowed_tools = set(allowed_tools)

    def check(self, tool_name: str) -> GovernanceDecision:
        allowed = tool_name in self.allowed_tools
        reason = "Tool in allowed list" if allowed else (
            f"Tool '{tool_name}' blocked by CapabilityGuard - "
            f"not in allowed list: {sorted(self.allowed_tools)}"
        )
        decision = GovernanceDecision(allowed=allowed, tool_name=tool_name, reason=reason)
        logger.info(f"CapabilityGuard: {tool_name} -> {'ALLOW' if allowed else 'DENY'}")
        return decision


class PolicyEngine:
    """Regex-based policy rules for input text (PII, sensitive data)."""

    def __init__(self, rules: list[dict] | None = None):
        self.rules = rules or []

    def check_input(self, text: str) -> GovernanceDecision:
        for rule in self.rules:
            pattern = rule.get("pattern", "")
            if re.search(pattern, text):
                reason = rule.get("message", f"Blocked by rule: {rule.get('name', 'unnamed')}")
                logger.warning(f"PolicyEngine DENY: {rule.get('name')} matched")
                return GovernanceDecision(allowed=False, tool_name="input_check", reason=reason)
        return GovernanceDecision(allowed=True, tool_name="input_check", reason="Input passed all policy checks")


class GovernanceLayer:
    """Combines CapabilityGuard + PolicyEngine into a single governance layer."""

    def __init__(self, allowed_tools: list[str], policy_rules: list[dict] | None = None):
        self.capability_guard = CapabilityGuard(allowed_tools)
        self.policy_engine = PolicyEngine(policy_rules)

    def check_tool(self, tool_name: str) -> GovernanceDecision:
        return self.capability_guard.check(tool_name)

    def check_input(self, text: str) -> GovernanceDecision:
        return self.policy_engine.check_input(text)
'''

(agent_dir / "governance.py").write_text(governance_py.strip(), encoding="utf-8")
utils.print_ok("Created governed-agent/governance.py")

# ---- main.py: Foundry hosted agent with governance ----
main_py = r'''import os
import asyncio
from typing import Annotated
from datetime import datetime, timezone as tz

from pydantic import Field
from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from agent_framework_foundry_hosting import ResponsesHostServer
from azure.identity import DefaultAzureCredential

from governance import GovernanceLayer


# ---- Governance Configuration ----
governance = GovernanceLayer(
    allowed_tools=["get_current_time", "get_weather"],
    # send_email is NOT in the allowed list => BLOCKED
    policy_rules=[
        {
            "name": "block-ssn",
            "pattern": r"\b\d{3}-\d{2}-\d{4}\b",
            "message": "PII detected (SSN pattern) - blocked for data protection",
        },
        {
            "name": "block-sensitive-keywords",
            "pattern": r"(?i)(password|credit.card|social.security)",
            "message": "Sensitive information detected - blocked for data protection",
        },
    ],
)


# ---- Tools (governed via CapabilityGuard) ----
@tool(approval_mode="never_require")
async def get_current_time(
    timezone: Annotated[str, Field(description="IANA timezone name, e.g. 'America/New_York', 'Europe/London'")]
) -> str:
    """Get the current date and time for a given timezone."""
    decision = governance.check_tool("get_current_time")
    if not decision.allowed:
        return f"[GOVERNANCE BLOCKED] {decision.reason}"
    import zoneinfo
    try:
        zone = zoneinfo.ZoneInfo(timezone)
        now = datetime.now(zone)
        return f"Current time in {timezone}: {now.strftime('%Y-%m-%d %H:%M:%S %Z')}"
    except Exception as e:
        return f"Could not get time for timezone '{timezone}': {e}"


@tool(approval_mode="never_require")
async def get_weather(
    location: Annotated[str, Field(description="City name, e.g. 'Seattle', 'London', 'Tokyo'")]
) -> str:
    """Get the current weather for a given location (simulated)."""
    decision = governance.check_tool("get_weather")
    if not decision.allowed:
        return f"[GOVERNANCE BLOCKED] {decision.reason}"
    import hashlib
    seed = hashlib.md5(f"{location}{datetime.now(tz.utc).strftime('%Y-%m-%d')}".encode()).hexdigest()
    conditions = ["sunny", "partly cloudy", "cloudy", "rainy", "windy", "snowy"]
    condition = conditions[int(seed[:2], 16) % len(conditions)]
    temp_c = 5 + (int(seed[2:4], 16) % 30)
    humidity = 30 + (int(seed[4:6], 16) % 50)
    return f"Weather in {location}: {condition}, {temp_c} deg C, humidity {humidity}%"


@tool(approval_mode="never_require")
async def send_email(
    to: Annotated[str, Field(description="Recipient email address")],
    body: Annotated[str, Field(description="Email body text")],
) -> str:
    """Send an email to a recipient."""
    decision = governance.check_tool("send_email")
    if not decision.allowed:
        return f"[GOVERNANCE BLOCKED] {decision.reason}"
    return f"Email sent to {to}"


async def setup():
    credential = DefaultAzureCredential()

    client = FoundryChatClient(
        project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        credential=credential,
    )

    # Enable OpenTelemetry -> Application Insights
    try:
        await client.configure_azure_monitor(enable_sensitive_data=False)
    except Exception as e:
        print(f"Warning: Could not configure Azure Monitor: {e}")

    agent = Agent(
        client=client,
        name="citadel-governed-agent",
        instructions=(
            "You are a governed customer service assistant. "
            "You can check the weather, tell the time, and send emails. "
            "If a tool returns a GOVERNANCE BLOCKED message, explain to the user "
            "that the action was denied by the governance policy and state the reason. "
            "Be concise and helpful."
        ),
        tools=[get_current_time, get_weather, send_email],
        default_options={"store": False},
    )

    return ResponsesHostServer(agent)


if __name__ == "__main__":
    server = asyncio.run(setup())
    server.run()
'''
(agent_dir / "main.py").write_text(main_py.strip(), encoding="utf-8")
utils.print_ok("Created governed-agent/main.py")

# ---- requirements.txt (no AGT packages - governance is self-contained) ----
requirements_txt = """\
agent-framework>=1.2.0
agent-framework-foundry-hosting>=1.0.0a260507
azure-identity>=1.25.0
azure-monitor-opentelemetry>=1.6.0
"""
(agent_dir / "requirements.txt").write_text(requirements_txt, encoding="utf-8")
utils.print_ok("Created governed-agent/requirements.txt")

# ---- Dockerfile ----
dockerfile = """\
FROM python:3.13-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY governance.py .
COPY main.py .

EXPOSE 8088

CMD ["python", "main.py"]
"""
(agent_dir / "Dockerfile").write_text(dockerfile, encoding="utf-8")
utils.print_ok("Created governed-agent/Dockerfile")

# ---- agent.yaml ----
agent_yaml = f"""\
kind: hosted
name: {governed_agent_name}
protocols:
  - protocol: responses
    version: 1.0.0
resources:
  cpu: "1"
  memory: 2Gi
environment_variables:
  - name: AZURE_AI_MODEL_DEPLOYMENT_NAME
    value: {governed_agent_model}
  - name: ENABLE_INSTRUMENTATION
    value: "true"
  - name: ENABLE_SENSITIVE_DATA
    value: "true"
  - name: OTEL_SERVICE_NAME
    value: {governed_agent_name}
"""
(agent_dir / "agent.yaml").write_text(agent_yaml, encoding="utf-8")
utils.print_ok("Created governed-agent/agent.yaml")

utils.print_info(f"\nAll governed agent source files written to: {agent_dir.resolve()}")

<a id='3'></a>
### 3️⃣ Build & Push Container Image

Uses `az acr build` to build the Docker image in the cloud and push it to ACR.

> `--no-logs` is required because Azure CLI on Windows crashes streaming build logs due to a cp1252 encoding bug. The build runs server-side regardless.

In [ ]:
# NOTE: --no-logs is required because Azure CLI on Windows crashes streaming
# build logs due to a cp1252 encoding bug (pip output contains Unicode chars
# that colorama can't encode). The build runs server-side regardless.
acr_build_cmd = (
    f'az acr build'
    f' --registry {env_acr_name}'
    f' --resource-group {env_foundry_resource_group}'
    f' --image {governed_agent_name}:{governed_agent_image_tag}'
    f' --file governed-agent/Dockerfile'
    f' --no-logs'
    f' governed-agent/'
)

utils.print_info(f"Building image in ACR: {governed_agent_image}")
utils.print_info("Build logs suppressed (Windows encoding bug). Polling for completion...")

output = utils.run(acr_build_cmd, "ACR build queued", "ACR build queue failed")

# Wait for the image to appear in ACR (build takes ~2-3 min)
import time as _t
for attempt in range(24):  # up to 4 minutes
    verify = utils.run(
        f'az acr repository show-tags --name {env_acr_name} --repository {governed_agent_name} --output json',
        "", ""
    )
    if verify.success and verify.json_data and governed_agent_image_tag in verify.json_data:
        utils.print_ok(f"Image verified in ACR: {governed_agent_image}")
        break
    utils.print_info(f"  Waiting for image... ({(attempt+1)*10}s)")
    _t.sleep(10)
else:
    utils.print_error(f"Image '{governed_agent_image_tag}' not found after 4 min. Check ACR build logs in the portal.")

<a id='4'></a>
### 4️⃣ Deploy Governed Agent to Foundry

Creates a new hosted agent version in the Foundry project using the Python SDK. This agent includes AGT governance — every tool call is checked against the `read_only` and `no_pii` policies before execution.

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    HostedAgentDefinition,
    ProtocolVersionRecord,
    AgentProtocol,
)
from azure.identity import DefaultAzureCredential

utils.print_info(f"Connecting to Foundry endpoint: {foundry_project_endpoint}")

credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=credential,
    allow_preview=True,
)
utils.print_ok(f"Foundry project client initialized for: {env_foundry_project_name}")

# Deploy the governed hosted agent
utils.print_info(f"Deploying governed agent: {governed_agent_name}")
utils.print_info(f"Image: {governed_agent_image}")
utils.print_info(f"Model: {governed_agent_model}")

agent = project_client.agents.create_version(
    agent_name=governed_agent_name,
    definition=HostedAgentDefinition(
        container_protocol_versions=[
            ProtocolVersionRecord(
                protocol=AgentProtocol.RESPONSES,
                version="1.0.0",
            )
        ],
        cpu="1",
        memory="2Gi",
        image=governed_agent_image,
        environment_variables={
            "AZURE_AI_MODEL_DEPLOYMENT_NAME": governed_agent_model,
            "ENABLE_INSTRUMENTATION": "true",
            "ENABLE_SENSITIVE_DATA": "true",
            "OTEL_SERVICE_NAME": governed_agent_name,
        },
    ),
)

utils.print_ok(f"Agent version created: {agent.name} (version: {agent.version})")
utils.print_info(f"Status: {agent.status}")

<a id='5'></a>
### 5️⃣ Wait for Agent to Become Active

Poll the agent version status until it becomes `active`. This typically takes 2-5 minutes as the platform pulls the image and starts the container.

In [ ]:
max_wait_seconds = 600  # 10 minutes
poll_interval = 15

elapsed = 0
while elapsed < max_wait_seconds:
    version = project_client.agents.get_version(
        agent_name=agent.name,
        agent_version=agent.version,
    )
    status = version.status
    utils.print_info(f"[{elapsed}s] Agent status: {status}")

    if status == "active":
        utils.print_ok(f"Agent is active! (took ~{elapsed}s)")
        break
    elif status in ("failed", "error"):
        utils.print_error(f"Agent deployment failed with status: {status}")
        break

    time.sleep(poll_interval)
    elapsed += poll_interval
else:
    utils.print_error(f"Timed out after {max_wait_seconds}s waiting for agent to become active.")

<a id='5a'></a>
### 5a. Assign RBAC to Agent Identity

Every hosted agent gets a dedicated Microsoft Entra ID (agent identity) at deploy time. This identity needs the **Foundry User** role on the Foundry account so it can access conversation storage and model endpoints.

In [ ]:
# Retrieve the agent identity assigned by the platform
agent_identity = version.instance_identity

if agent_identity is None:
    agent_details = project_client.agents.get(agent_name=governed_agent_name)
    agent_identity = agent_details.instance_identity

if agent_identity is None:
    utils.print_error("Could not resolve agent identity. Ensure the agent is in 'active' state.")
else:
    agent_principal_id = agent_identity.principal_id
    utils.print_ok(f"Agent identity principal ID: {agent_principal_id}")

    foundry_account_scope = (
        f"/subscriptions/{env_foundry_subscription_id}"
        f"/resourceGroups/{env_foundry_resource_group}"
        f"/providers/Microsoft.CognitiveServices/accounts/{env_foundry_account_name}"
    )

    # Check if Foundry User role is already assigned
    check = subprocess.run(
        ["az", "role", "assignment", "list",
         "--assignee-object-id", agent_principal_id,
         "--role", "Foundry User",
         "--scope", foundry_account_scope,
         "--query", "[0].id", "-o", "tsv"],
        capture_output=True, text=True, shell=True,
    )

    if check.stdout.strip():
        utils.print_ok("Foundry User role is already assigned.")
    else:
        utils.print_info(f"Assigning 'Foundry User' to agent identity on: {env_foundry_account_name}")
        result = subprocess.run(
            ["az", "role", "assignment", "create",
             "--assignee-object-id", agent_principal_id,
             "--assignee-principal-type", "ServicePrincipal",
             "--role", "Foundry User",
             "--scope", foundry_account_scope,
             "--only-show-errors", "-o", "none"],
            capture_output=True, text=True, shell=True,
        )
        if result.returncode == 0:
            utils.print_ok("Foundry User role assigned. Waiting 60s for RBAC propagation...")
            time.sleep(60)
            utils.print_ok("Done - RBAC should be active now.")
        else:
            utils.print_error(f"Failed to assign role: {result.stderr}")

<a id='6'></a>
### 6️⃣ Test Governance: Allowed Actions

Test tools that the `read_only` policy **allows** — `get_weather` and `get_current_time` are read-only operations and should execute normally.

In [ ]:
import uuid

# Get an OpenAI client scoped to the governed agent
openai_client = project_client.get_openai_client(agent_name=governed_agent_name)

conversation_id = str(uuid.uuid4())
utils.print_info(f"Conversation ID: {conversation_id}")

# Test 1: Weather query (ALLOWED - read-only action)
utils.print_info("\nTest 1: What's the weather in Seattle? (should be ALLOWED)")
response = openai_client.responses.create(
    input="What's the weather in Seattle?",
    metadata={"conversation_id": conversation_id},
)
utils.print_ok(f"Agent: {response.output_text}")

# Test 2: Time query (ALLOWED - read-only action)
utils.print_info("\nTest 2: What time is it in Tokyo? (should be ALLOWED)")
response2 = openai_client.responses.create(
    input="What time is it in Tokyo?",
    metadata={"conversation_id": conversation_id},
)
utils.print_ok(f"Agent: {response2.output_text}")

<a id='7'></a>
### 7️⃣ Test Governance: Blocked Actions

Test tools that the `read_only` policy **blocks** — `send_email` is a write action and should be denied by AGT. The agent should explain the governance denial to the user.

In [ ]:
# Test 3: Send email (BLOCKED by read_only policy)
utils.print_info("Test 3: Send an email to user@contoso.com (should be BLOCKED)")
response3 = openai_client.responses.create(
    input="Send an email to user@contoso.com saying 'Hello from the governed agent'",
    metadata={"conversation_id": conversation_id},
)
utils.print_ok(f"Agent: {response3.output_text}")

# Verify the response mentions the block
if "BLOCKED" in response3.output_text.upper() or "denied" in response3.output_text.lower() or "policy" in response3.output_text.lower():
    utils.print_ok("Governance correctly blocked the write action!")
else:
    utils.print_warning("Expected governance block message not detected in response.")

<a id='8'></a>
### 8️⃣ Local Governance Simulation

Run the AGT `CapabilityGuardMiddleware` **locally** in this notebook to demonstrate the same allow/block decisions that happen server-side inside the container.

This is useful for:
- Testing which tools are allowed/blocked before deploying
- Understanding the capability guard allow-list pattern
- Rapid iteration on governance policies

In [ ]:
# Import the same governance module used inside the container
import importlib.util, pathlib
spec = importlib.util.spec_from_file_location("governance", pathlib.Path("governed-agent/governance.py"))
gov_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gov_module)

# Same config as the hosted agent container
local_governance = gov_module.GovernanceLayer(
    allowed_tools=["get_current_time", "get_weather"],
    policy_rules=[
        {"name": "block-ssn", "pattern": r"\b\d{3}-\d{2}-\d{4}\b", "message": "PII detected (SSN pattern)"},
        {"name": "block-sensitive", "pattern": r"(?i)(password|credit.card|social.security)", "message": "Sensitive info detected"},
    ],
)

utils.print_info("GovernanceLayer initialized (same config as container)")
utils.print_info(f"Allowed tools: {sorted(local_governance.capability_guard.allowed_tools)}")


def test_tool(tool_name: str, description: str):
    """Test if a tool call would be allowed by the capability guard."""
    decision = local_governance.check_tool(tool_name)
    icon = "\u2705" if decision.allowed else "\u274c"
    status = "ALLOW" if decision.allowed else "DENY"
    print(f"  {icon} {status}  {tool_name} \u2014 {description}")
    if not decision.allowed:
        print(f"         Reason: {decision.reason}")
    return decision


def test_input(text: str, description: str):
    """Test if input text passes policy checks."""
    decision = local_governance.check_input(text)
    icon = "\u2705" if decision.allowed else "\u274c"
    status = "ALLOW" if decision.allowed else "DENY"
    print(f"  {icon} {status}  \"{text[:50]}\" \u2014 {description}")
    if not decision.allowed:
        print(f"         Reason: {decision.reason}")
    return decision


print("=" * 60)
print("AGT GOVERNANCE SIMULATION (GovernanceLayer)")
print(f"Allowed tools: {sorted(local_governance.capability_guard.allowed_tools)}")
print("=" * 60)

print("\n--- CAPABILITY GUARD (tool allow-list) ---")

# Test 1: Allowed
print("\n[Test 1] Weather lookup (in allowed_tools):")
test_tool("get_weather", "read-only weather lookup")

# Test 2: Allowed
print("\n[Test 2] Time lookup (in allowed_tools):")
test_tool("get_current_time", "read-only time lookup")

# Test 3: Blocked
print("\n[Test 3] Send email (NOT in allowed_tools):")
test_tool("send_email", "write action")

# Test 4: Blocked
print("\n[Test 4] File write (NOT in allowed_tools):")
test_tool("file_write", "write action")

# Test 5: Blocked
print("\n[Test 5] Database write (NOT in allowed_tools):")
test_tool("database_write", "destructive action")

print("\n--- POLICY ENGINE (input text rules) ---")

# Test 6: Blocked - SSN pattern
print("\n[Test 6] SSN in user message:")
test_input("My SSN is 123-45-6789, look up my account", "PII pattern detected")

# Test 7: Blocked - sensitive keyword
print("\n[Test 7] Password mention:")
test_input("My password is hunter2", "sensitive keyword detected")

# Test 8: Allowed - normal query
print("\n[Test 8] Normal weather query:")
test_input("What is the weather in Seattle?", "clean input")

<a id='9'></a>
### 9️⃣ Governance Summary

| Layer | Component | Where it runs |
|-------|-----------|---------------|
| **This notebook** | Client + governance sim | Your machine |
| **Foundry** | Hosted Agent container | Azure |
| **AGT** (Layer 2) | `StatelessKernel` + policies | Inside the container |
| **Citadel** (Layer 1) | Event Hub + Governance Hub | Azure |

**Results:**
- `get_weather` and `get_current_time` — **ALLOWED** (in `allowed_tools` list)
- `send_email`, `file_write`, `database_write` — **BLOCKED** by `CapabilityGuardMiddleware` (not in allowed list)
- PII/sensitive content — **BLOCKED** by `GovernancePolicyMiddleware` YAML rules
- All decisions are logged by `AuditTrailMiddleware` for compliance

**Next steps:**
- Export governance events to Citadel Event Hub for centralized auditing
- Customize policies (add your own rules to `StatelessKernel`)
- Enable trust scoring for continuous agent monitoring

<a id='10'></a>
### 🔟 Cleanup

Delete the governed agent entirely. Set `cleanup_enabled = True` to remove the agent.

> **Note:** This removes the agent and its Entra ID identity from the Foundry project. The ACR image and spoke resources are **not** affected.

In [ ]:
cleanup_enabled = False  # Set to True to enable cleanup

if cleanup_enabled:
    try:
        result = project_client.agents.delete(agent_name=agent.name)
        utils.print_ok(f"Agent deleted: {agent.name} (deleted={result.get('deleted', True)})")
    except Exception as e:
        utils.print_warning(f"Failed to delete agent: {e}")
else:
    utils.print_info(f"Cleanup is disabled. Set cleanup_enabled = True to remove the agent.")
    utils.print_info(f"Agent retained: {agent.name} (version: {agent.version})")